---
title: "Lab: Mini-Lakehouse with Delta Lake"
format: html
---


# Day 1 — Modern Data Engineering Fundamentals & Mini-Lakehouse

This single notebook covers everything in today's lab:

1. **ETL vs ELT, and Warehouse vs Lake vs Lakehouse** — a runnable comparison of the three
   ingestion/storage patterns named in the course objectives.
2. **Mini-Lakehouse with Delta Lake** — a hands-on PySpark + Delta Lake demo proving ACID
   transactions, schema enforcement, time travel, `OPTIMIZE`/Z-ORDER, and schema evolution.


In [1]:
!pip install pyspark==3.5.0 delta-spark==3.2.0


## Part 1 — ETL vs ELT, and Warehouse vs Lake vs Lakehouse

Loads the same raw e-commerce order batch three different ways so the contrast between the
three patterns is visible in the output, not just described in words: ETL (clean-before-load,
warehouse style), ELT (load-before-clean, lake style), and the Lakehouse pattern (raw Bronze
landing + governed Silver transform) that Part 2 then builds on with Delta Lake.


In [2]:
import os
import shutil

import pandas as pd

RAW_ORDERS = [
    {"order_id": "ORD001", "user_id": "User_A", "product": "Laptop", "price": "1200.00", "quantity": "1"},
    {"order_id": "ORD002", "user_id": "User_B", "product": "Mouse", "price": "25.5", "quantity": "2"},
    {"order_id": "ORD003", "user_id": "User_C", "product": None, "price": "75.00", "quantity": "1"},  # bad row: missing product
    {"order_id": "ORD004", "user_id": "User_A", "product": "Headphones", "price": "not_a_number", "quantity": "1"},  # bad row: corrupt price
]


def clean_and_validate(rows):
    """Shared cleaning rule used by the ETL path up front, and by the ELT/Lakehouse paths later."""
    good, rejected = [], []
    for row in rows:
        if not row["product"]:
            rejected.append((row, "missing product"))
            continue
        try:
            price = float(row["price"])
        except ValueError:
            rejected.append((row, "corrupt price"))
            continue
        good.append({**row, "price": price, "quantity": int(row["quantity"])})
    return good, rejected


def etl_style_warehouse_load():
    """
    ETL ("Transform" happens BEFORE "Load"):
    Classic data-warehouse pattern. Bad rows are cleaned/rejected in-flight,
    by the ingestion job itself, before anything ever lands in storage.
    Storage only ever sees clean, schema-conforming rows.
    """
    print("\n=== ETL (transform-before-load)  -  Data Warehouse style ===")
    good, rejected = clean_and_validate(RAW_ORDERS)
    print(f"Rows accepted into the warehouse table: {len(good)}")
    print(f"Rows rejected before ever reaching storage: {len(rejected)}")
    for row, reason in rejected:
        print(f"  dropped {row['order_id']}: {reason}")
    warehouse_table = pd.DataFrame(good)
    print(warehouse_table)
    print("Trade-off: storage is always clean, but the transform logic is baked into the")
    print("ingestion job  -  changing a business rule means re-running ingestion from source.")
    return warehouse_table


def elt_style_lake_load():
    """
    ELT ("Load" happens BEFORE "Transform"):
    Classic data-lake pattern. Everything lands as-is, including bad rows  - 
    there is no schema or quality gate at write time. Transformation is a
    separate, later step run against whatever was dumped.
    """
    print("\n=== ELT (load-before-transform)  -  Data Lake style ===")
    raw_table = pd.DataFrame(RAW_ORDERS)
    print(f"Rows landed in the raw lake zone (no gate at all): {len(raw_table)}")
    print(raw_table)
    print("Transforming later, on read, against whatever is sitting in the lake...")
    good, rejected = clean_and_validate(RAW_ORDERS)
    print(f"Only discovered at transform time that {len(rejected)} row(s) were bad  - ")
    print("they already consumed storage and could have been queried by someone else first.")
    print("Trade-off: ingestion is fast and never blocks on data quality, but the raw zone")
    print("can silently become a 'data swamp' if nothing ever comes back to clean it up.")
    return raw_table


def lakehouse_style_load(tmp_path="./data/etl_elt_demo"):
    """
    Lakehouse: gets ELT's speed (raw data lands immediately, nothing is
    dropped at ingestion) AND ETL's reliability (a governed table format
    enforces schema and keeps an auditable transaction log), by separating
    "raw landing" from "governed table" into Bronze/Silver layers instead of
    forcing that trade-off into ingestion time or read time.
    """
    print("\n=== Lakehouse (raw landing + governed transform) ===")
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    os.makedirs(tmp_path, exist_ok=True)

    bronze = pd.DataFrame(RAW_ORDERS)
    bronze_path = os.path.join(tmp_path, "bronze_orders.parquet")
    bronze.to_parquet(bronze_path)
    print(f"Bronze (raw, everything lands as-is, {len(bronze)} rows) written to {bronze_path}")

    good, rejected = clean_and_validate(RAW_ORDERS)
    silver = pd.DataFrame(good)
    silver_path = os.path.join(tmp_path, "silver_orders.parquet")
    silver.to_parquet(silver_path)
    print(f"Silver (governed, schema-checked, {len(silver)} rows, {len(rejected)} quarantined)")
    print("written to", silver_path)
    print("Bronze is never lost, so a bad Silver rule can always be re-run from Bronze  - ")
    print("this is what Day 1's Delta Lake lab then adds ACID/versioning/time-travel on top of.")
    return bronze, silver


def main():
    warehouse_table = etl_style_warehouse_load()
    lake_table = elt_style_lake_load()
    bronze, silver = lakehouse_style_load()

    print("\n=== Summary ===")
    print(f"{'Pattern':<12}{'Rows at rest':<15}{'Quality gate timing':<25}")
    print(f"{'Warehouse':<12}{len(warehouse_table):<15}{'before load (ETL)':<25}")
    print(f"{'Lake':<12}{len(lake_table):<15}{'on read, later (ELT)':<25}")
    print(f"{'Lakehouse':<12}{len(bronze)} raw / {len(silver)} governed   {'both: raw at load, governed after':<25}")


In [3]:
main()



=== ETL (transform-before-load)  -  Data Warehouse style ===
Rows accepted into the warehouse table: 2
Rows rejected before ever reaching storage: 2
  dropped ORD003: missing product
  dropped ORD004: corrupt price
  order_id user_id product   price  quantity
0   ORD001  User_A  Laptop  1200.0         1
1   ORD002  User_B   Mouse    25.5         2
Trade-off: storage is always clean, but the transform logic is baked into the
ingestion job  -  changing a business rule means re-running ingestion from source.

=== ELT (load-before-transform)  -  Data Lake style ===
Rows landed in the raw lake zone (no gate at all): 4
  order_id user_id     product         price quantity
0   ORD001  User_A      Laptop       1200.00        1
1   ORD002  User_B       Mouse          25.5        2
2   ORD003  User_C         NaN         75.00        1
3   ORD004  User_A  Headphones  not_a_number        1
Transforming later, on read, against whatever is sitting in the lake...
Only discovered at transform time th

## Part 2 — Mini-Lakehouse with Delta Lake

Everything below is the annotated walkthrough of a local Delta Lake mini-lakehouse: ACID
transactions, schema enforcement, time travel, `OPTIMIZE`/Z-ORDER, and schema evolution.


In [4]:
!pip install delta-spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.1 MB/s eta 0:00:00


In [5]:
import glob
import json
import os
import shutil

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StructField,
    StructType,
    StringType,
)


# ─────────────────────────────────────────────────────────────────────────────
# SESSION SETUP
# ─────────────────────────────────────────────────────────────────────────────

def create_spark_session() -> SparkSession:
    builder = (
        SparkSession.builder
        .appName("MiniLakehouse_Extended")
        .master("local[*]")
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.warehouse.dir", "./data/warehouse")
        # Enable auto-optimization (Delta Lake 2.x+)
        .config("spark.databricks.delta.optimizeWrite.enabled", "true")
        .config("spark.databricks.delta.autoCompact.enabled",   "true")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()


ORDER_SCHEMA = StructType([
    StructField("order_id",  StringType(),  nullable=False),
    StructField("user_id",   StringType(),  nullable=True),
    StructField("product",   StringType(),  nullable=True),
    StructField("price",     DoubleType(),  nullable=True),
    StructField("quantity",  IntegerType(), nullable=True),
    StructField("region",    StringType(),  nullable=True),
])


def sep(label: str) -> None:
    print(f"\n{'=' * 65}")
    print(f"  {label}")
    print("=" * 65)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    TABLE_PATH = "./data/delta/orders"

    if os.path.exists("./data"):
        shutil.rmtree("./data")

    print("\nBooting Spark session with Delta Lake engine...")
    spark = create_spark_session()
    spark.sparkContext.setLogLevel("ERROR")

    # ── STEP 1: Initial Write ─────────────────────────────────────────────────
    sep("STEP 1 — Initial Batch Ingestion (WRITE, partitioned by region)")

    batch_1 = [
        ("ORD001", "usr_alice",  "Laptop",      1200.00, 1, "MENA"),
        ("ORD002", "usr_bob",    "Mouse",          25.50, 2, "EU"),
        ("ORD003", "usr_carol",  "Keyboard",       75.00, 1, "US"),
        ("ORD004", "usr_alice",  "Headphones",    150.00, 1, "MENA"),
        ("ORD005", "usr_dave",   "Monitor",       300.00, 2, "EU"),
    ]
    df1 = spark.createDataFrame(batch_1, ORDER_SCHEMA)
    (df1.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("region")     # Physical partitioning — Spark skips entire
        .save(TABLE_PATH))         # partition directories during filtered queries.

    print("Delta table written and partitioned by 'region'.")
    spark.read.format("delta").load(TABLE_PATH).show()

    # ── STEP 2: ACID Append ───────────────────────────────────────────────────
    sep("STEP 2 — ACID Append Transaction")

    batch_2 = [
        ("ORD006", "usr_eve",   "Tablet",   200.00, 1, "US"),
        ("ORD007", "usr_bob",   "Webcam",    80.00, 3, "EU"),
    ]
    df2 = spark.createDataFrame(batch_2, ORDER_SCHEMA)
    df2.write.format("delta").mode("append").save(TABLE_PATH)

    print("Appended 2 new rows atomically. All-or-nothing guarantee enforced.")
    spark.read.format("delta").load(TABLE_PATH).show()

    # ── STEP 3: Schema Enforcement ────────────────────────────────────────────
    sep("STEP 3 — Schema Enforcement (Reject Malformed Ingestion)")
    print("Attempting to write a row with an undeclared 'discount' column...")

    bad_schema = StructType([
        StructField("order_id",  StringType(),  True),
        StructField("user_id",   StringType(),  True),
        StructField("product",   StringType(),  True),
        StructField("price",     DoubleType(),  True),
        StructField("quantity",  IntegerType(), True),
        StructField("region",    StringType(),  True),
        StructField("discount",  DoubleType(),  True),  # Not in the registered schema
    ])
    df_bad = spark.createDataFrame(
        [("ORD999", "usr_hacker", "Free Item", 0.0, 99, "MENA", 100.0)],
        bad_schema,
    )
    try:
        df_bad.write.format("delta").mode("append").save(TABLE_PATH)
    except Exception as e:
        print("Schema violation caught by Delta Lake — write REJECTED.")
        print(f"  Error: {str(e).splitlines()[0]}")
        print("\nWhy this matters: without schema enforcement a single misconfigured")
        print("upstream pipeline can silently corrupt a production lakehouse table.")

    # ── STEP 4: MERGE / UPSERT ────────────────────────────────────────────────
    sep("STEP 4 — MERGE / UPSERT (Atomic Update + Insert in One Transaction)")
    print("""
MERGE is the most powerful Delta operation — impossible in a raw data lake:
  • If a matching record exists  → UPDATE specific columns
  • If no matching record        → INSERT the new row
  • All in one atomic transaction (either all commits or nothing commits)

Use case: a nightly CDC (Change Data Capture) feed from a production database
that may contain both price corrections and new orders in the same payload.
""")
    upsert_data = [
        ("ORD002", "usr_bob",    "Mouse",       29.99, 2, "EU"),   # exists → update price
        ("ORD010", "usr_frank",  "SSD Drive",   95.00, 1, "US"),   # new    → insert
    ]
    df_upsert = spark.createDataFrame(upsert_data, ORDER_SCHEMA)

    delta_table = DeltaTable.forPath(spark, TABLE_PATH)
    (
        delta_table.alias("target")
        .merge(
            df_upsert.alias("updates"),
            "target.order_id = updates.order_id",
        )
        .whenMatchedUpdate(set={
            "price":    "updates.price",
            "quantity": "updates.quantity",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE complete. ORD002 price updated to 29.99; ORD010 inserted.")
    spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()

    # ── STEP 5: OPTIMIZE + Z-ORDER ────────────────────────────────────────────
    sep("STEP 5 — OPTIMIZE + Z-ORDER BY (Data Skipping for Query Acceleration)")
    print("""
The small-file problem:
  Streaming micro-batches and per-partition writes create thousands of tiny
  Parquet files. Each file read requires a separate I/O operation — at scale
  this kills query performance.

OPTIMIZE:
  Merges small files into fewer, larger Parquet files, reducing metadata
  overhead by up to 99%.

Z-ORDER BY:
  Physically co-locates rows with similar values in the same Parquet file.
  Delta reads the _delta_log/ min/max statistics per file. If your query is
  WHERE user_id = 'usr_alice', Spark skips all files that cannot contain that
  user — called data skipping.

Benchmark: Z-Ordering a 1 TB table on user_id typically reduces query time
from 4 minutes → 8 seconds on filtered queries.
""")
    # spark.sql(f"OPTIMIZE delta.`{TABLE_PATH}` ZORDER BY (user_id, region)") # Original line causing error
    delta_table.optimize().executeZOrderBy(["user_id"])
    print("OPTIMIZE + ZORDER complete.")

    # ── STEP 6: Delta Transaction Log Inspection ──────────────────────────────
    sep("STEP 6 — Inspecting the Delta Transaction Log (_delta_log/)")
    print("""
Every Delta write appends a JSON file to _delta_log/.
This is the single source of truth for ACID compliance, Time Travel, and lineage.
Each JSON file contains one or more entries:
  commitInfo  → operation type, timestamp, user, operation parameters
  add         → new Parquet file paths written in this commit
  remove      → old Parquet file paths superseded by this commit
  metaData    → schema changes (only present when schema mutates)

After every 10 commits Delta writes a checkpoint.parquet to avoid replaying
thousands of JSON files on startup — this is the Delta Log compaction process.
""")
    log_files = sorted(glob.glob(f"{TABLE_PATH}/_delta_log/*.json"))
    for log_path in log_files[:4]:
        print(f"\n--- {os.path.basename(log_path)} ---")
        with open(log_path, encoding="utf-8") as fh:
            for line in fh:
                entry = json.loads(line)
                if "commitInfo" in entry:
                    info = entry["commitInfo"]
                    print(f"  operation  : {info.get('operation', 'N/A')}")
                    print(f"  timestamp  : {info.get('timestamp', 'N/A')}")
                    params = info.get("operationParameters", {})
                    if params:
                        print(f"  parameters : {params}")
                elif "add" in entry:
                    print(f"  add file   : {entry['add'].get('path', '')}")
                elif "remove" in entry:
                    print(f"  remove file: {entry['remove'].get('path', '')}")

    # ── STEP 7: Time Travel ───────────────────────────────────────────────────
    sep("STEP 7 — Time Travel (Query Historical Versions)")

    print("Full audit trail from the Delta Transaction Log:")
    delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

    print("Restoring state as of VERSION 0 (initial ingest only):")
    spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH).show()

    print("Restoring state as of VERSION 1 (after first ACID append):")
    spark.read.format("delta").option("versionAsOf", 1).load(TABLE_PATH).show()

    # ── STEP 8: VACUUM ────────────────────────────────────────────────────────
    sep("STEP 8 — VACUUM (Storage Retention Cleanup)")
    print("""
Delta retains all historical Parquet files to support Time Travel.
VACUUM permanently deletes files older than the retention window.

WARNING: After VACUUM, Time Travel beyond the retention window is impossible.

Production recommendation: retentionHours >= 168 (7 days).
Under GDPR Article 17 (right to erasure), lower retention may be required —
but confirm with your legal team before reducing below 168 hours.

Setting 0 hours here for demonstration only — NEVER do this in production.
""")
    spark.conf.set(
        "spark.databricks.delta.retentionDurationCheck.enabled", "false"
    )
    delta_table.vacuum(retentionHours=0)
    print("VACUUM complete. Orphaned files removed.")

    # ── STEP 9: Schema Evolution ──────────────────────────────────────────────
    sep("STEP 9 — Schema Evolution (Safe Column Addition via mergeSchema)")
    print("""
Scenario: a product team wants to add a 'discount' column to the orders feed.

Without Schema Evolution: the write would be REJECTED (Step 3 behaviour).
With mergeSchema=True:   the new column is added to the registered schema.
                         Existing rows receive NULL for the new column.

This is the production-safe migration path:
  1. Data contract teams agree on the new column definition.
  2. Producer writes first batch with mergeSchema=True.
  3. Delta propagates the schema change; all downstream readers automatically
     see the new nullable column on their next read.
""")
    evolved_schema = StructType([
        StructField("order_id",  StringType(),  True),
        StructField("user_id",   StringType(),  True),
        StructField("product",   StringType(),  True),
        StructField("price",     DoubleType(),  True),
        StructField("quantity",  IntegerType(), True),
        StructField("region",    StringType(),  True),
        StructField("discount",  DoubleType(),  True),  # new column
    ])
    df_evolved = spark.createDataFrame(
        [("ORD011", "usr_grace", "SmartWatch", 250.00, 1, "MENA", 15.0)],
        evolved_schema,
    )
    (df_evolved.write
               .format("delta")
               .mode("append")
               .option("mergeSchema", "true")
               .save(TABLE_PATH))

    print("New column 'discount' merged into existing schema without downtime.")
    print("Pre-evolution rows show NULL for 'discount' — backward compatible:")
    spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()

    spark.stop()
    print("\nMini Lakehouse lab complete. All 9 steps executed successfully.")


if __name__ == "__main__":
    main()


Booting Spark session with Delta Lake engine...

  STEP 1 — Initial Batch Ingestion (WRITE, partitioned by region)
Delta table written and partitioned by 'region'.
+--------+---------+----------+------+--------+------+
|order_id|  user_id|   product| price|quantity|region|
+--------+---------+----------+------+--------+------+
|  ORD001|usr_alice|    Laptop|1200.0|       1|  MENA|
|  ORD004|usr_alice|Headphones| 150.0|       1|  MENA|
|  ORD003|usr_carol|  Keyboard|  75.0|       1|    US|
|  ORD002|  usr_bob|     Mouse|  25.5|       2|    EU|
|  ORD005| usr_dave|   Monitor| 300.0|       2|    EU|
+--------+---------+----------+------+--------+------+


  STEP 2 — ACID Append Transaction
Appended 2 new rows atomically. All-or-nothing guarantee enforced.
+--------+---------+----------+------+--------+------+
|order_id|  user_id|   product| price|quantity|region|
+--------+---------+----------+------+--------+------+
|  ORD001|usr_alice|    Laptop|1200.0|       1|  MENA|
|  ORD004|usr_al

Let's start by explaining the import statements and then move to the function definitions.

### Imports

*   `import glob`, `import json`, `import os`, `import shutil`: These lines import standard Python modules for working with file paths (`glob`, `os`), JSON data (`json`), and high-level file operations like deleting directories (`shutil`).
*   `from delta import configure_spark_with_delta_pip`: This imports a specific function from the `delta` library to simplify setting up a Spark session with Delta Lake capabilities.
*   `from delta.tables import DeltaTable`: This imports the `DeltaTable` class, which is used to interact with Delta Lake tables programmatically.
*   `from pyspark.sql import SparkSession`: Imports the main entry point for Spark functionality.
*   `from pyspark.sql.types import (...)`: Imports various data types (`DoubleType`, `IntegerType`, `StructField`, `StructType`, `StringType`) from PySpark's SQL module, which are used to define the schema of DataFrames.

### SESSION SETUP

This section defines functions and schema that are crucial for setting up the Spark environment and defining the data structure.

#### `create_spark_session()` function

This function is responsible for creating and configuring a SparkSession with Delta Lake extensions.

```python
def create_spark_session() -> SparkSession:
    builder = (
        SparkSession.builder
        .appName("MiniLakehouse_Extended")
        .master("local[*]")
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.warehouse.dir", "./data/warehouse")
        .config("spark.databricks.delta.optimizeWrite.enabled", "true")
        .config("spark.databricks.delta.autoCompact.enabled",   "true")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()
```

*   `def create_spark_session() -> SparkSession:`: Defines a function named `create_spark_session` that returns a `SparkSession` object.
*   `builder = (...)`: Initializes a `SparkSession.builder` object to configure various Spark settings.
*   `.appName("MiniLakehouse_Extended")`: Sets the name of the Spark application.
*   `.master("local[*]")`: Configures Spark to run in local mode, using all available CPU cores.
*   `.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")`: Enables Delta Lake SQL functionalities.
*   `.config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")`: Registers Delta Lake as the default catalog for Spark SQL, allowing it to manage Delta tables.
*   `.config("spark.sql.warehouse.dir", "./data/warehouse")`: Specifies the directory where Spark will store its warehouse data.
*   `.config("spark.databricks.delta.optimizeWrite.enabled", "true")`: Enables automatic optimization during write operations, which helps in merging small files.
*   `.config("spark.databricks.delta.autoCompact.enabled", "true")`: Enables automatic compaction of small files to improve read performance.
*   `return configure_spark_with_delta_pip(builder).getOrCreate()`: Applies the Delta Lake configurations and then creates and returns the `SparkSession`.

#### `ORDER_SCHEMA` definition

This defines the structure of the data that will be stored in the Delta table.

```python
ORDER_SCHEMA = StructType([
    StructField("order_id",  StringType(),  nullable=False),
    StructField("user_id",   StringType(),  nullable=True),
    StructField("product",   StringType(),  nullable=True),
    StructField("price",     DoubleType(),  nullable=True),
    StructField("quantity",  IntegerType(), nullable=True),
    StructField("region",    StringType(),  nullable=True),
])
```

*   `ORDER_SCHEMA = StructType([...])`: Defines a `StructType` which represents the schema of a DataFrame. It's a list of `StructField` objects.
*   `StructField("order_id", StringType(), nullable=False)`: Defines a column named `order_id` of type `StringType`, which cannot be null.
*   Similarly, other `StructField` entries define `user_id`, `product`, `price`, `quantity`, and `region` with their respective types and nullability settings. This schema ensures data consistency in the Delta table.

#### `sep(label: str)` function

This is a simple helper function for printing separators in the console output to make the different steps more readable.

```python
def sep(label: str) -> None:
    print(f"\n{'=' * 65}")
    print(f"  {label}")
    print("=" * 65)
```

*   `def sep(label: str) -> None:`: Defines a function named `sep` that takes a string `label` and returns nothing (`None`).
*   `print(f"\n{'=' * 65}")`: Prints a line of 65 equal signs, followed by a newline, to create a visual separator.
*   `print(f"  {label}")`: Prints the provided `label` centered between the lines.
*   `print("=" * 65)`: Prints another line of 65 equal signs.

### MAIN Function

This is the core of the script, demonstrating various Delta Lake functionalities step by step.

```python
def main() -> None:
    TABLE_PATH = "./data/delta/orders"

    if os.path.exists("./data"):
        shutil.rmtree("./data")

    print("\nBooting Spark session with Delta Lake engine...")
    spark = create_spark_session()
    spark.sparkContext.setLogLevel("ERROR")
```

*   `def main() -> None:`: Defines the main function of the script.
*   `TABLE_PATH = "./data/delta/orders"`: Sets a variable `TABLE_PATH` to define the location where the Delta table will be stored.
*   `if os.path.exists("./data"): shutil.rmtree("./data")`: Checks if a directory named `data` exists. If it does, it deletes the entire directory and its contents to ensure a clean start for the demonstration.
*   `print("\nBooting Spark session with Delta Lake engine...")`: Prints a message indicating that the Spark session is starting.
*   `spark = create_spark_session()`: Calls the `create_spark_session` function (explained above) to initialize a SparkSession and assigns it to the `spark` variable.
*   `spark.sparkContext.setLogLevel("ERROR")`: Sets the logging level for Spark to `ERROR`, meaning only error messages will be displayed, reducing verbose output.

#### STEP 1: Initial Batch Ingestion (WRITE, partitioned by region)

This step demonstrates how to create a new Delta table and write initial data to it, partitioned by the 'region' column.

```python
    sep("STEP 1 — Initial Batch Ingestion (WRITE, partitioned by region)")

    batch_1 = [
        ("ORD001", "usr_alice",  "Laptop",      1200.00, 1, "MENA"),
        ("ORD002", "usr_bob",    "Mouse",          25.50, 2, "EU"),
        ("ORD003", "usr_carol",  "Keyboard",       75.00, 1, "US"),
        ("ORD004", "usr_alice",  "Headphones",    150.00, 1, "MENA"),
        ("ORD005", "usr_dave",   "Monitor",       300.00, 2, "EU"),
    ]
    df1 = spark.createDataFrame(batch_1, ORDER_SCHEMA)
    (df1.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("region")
        .save(TABLE_PATH))

    print("Delta table written and partitioned by 'region'.")
    spark.read.format("delta").load(TABLE_PATH).show()
```

*   `sep(...)`: Prints a separator with the step title.
*   `batch_1 = [...]`: Defines a list of tuples, where each tuple represents a row of data for the first batch.
*   `df1 = spark.createDataFrame(batch_1, ORDER_SCHEMA)`: Creates a Spark DataFrame `df1` from `batch_1` using the predefined `ORDER_SCHEMA`.
*   `(df1.write.format("delta").mode("overwrite").partitionBy("region").save(TABLE_PATH))`: This chain of methods performs the write operation:
    *   `.format("delta")`: Specifies that the data should be written in Delta Lake format.
    *   `.mode("overwrite")`: If the table already exists at `TABLE_PATH`, it will be completely overwritten.
    *   `.partitionBy("region")`: Organizes the data physically on disk into subdirectories based on the 'region' column's values. This optimizes queries that filter by 'region'.
    *   `.save(TABLE_PATH)`: Writes the DataFrame to the specified `TABLE_PATH`.
*   `print(...)`: Prints a confirmation message.
*   `spark.read.format("delta").load(TABLE_PATH).show()`: Reads the newly created Delta table and displays its contents.

#### STEP 2: ACID Append Transaction

This step demonstrates how to append new data to an existing Delta table, leveraging Delta Lake's ACID (Atomicity, Consistency, Isolation, Durability) properties.

```python
    sep("STEP 2 — ACID Append Transaction")

    batch_2 = [
        ("ORD006", "usr_eve",   "Tablet",   200.00, 1, "US"),
        ("ORD007", "usr_bob",   "Webcam",    80.00, 3, "EU"),
    ]
    df2 = spark.createDataFrame(batch_2, ORDER_SCHEMA)
    df2.write.format("delta").mode("append").save(TABLE_PATH)

    print("Appended 2 new rows atomically. All-or-nothing guarantee enforced.")
    spark.read.format("delta").load(TABLE_PATH).show()
```

*   `sep(...)`: Prints a separator with the step title.
*   `batch_2 = [...]`: Defines a list of tuples for the second batch of data.
*   `df2 = spark.createDataFrame(batch_2, ORDER_SCHEMA)`: Creates a Spark DataFrame `df2` from `batch_2` using the `ORDER_SCHEMA`.
*   `df2.write.format("delta").mode("append").save(TABLE_PATH)`: Appends `df2` to the existing Delta table at `TABLE_PATH`. `mode("append")` ensures that new data is added without affecting existing data. This is an atomic operation, meaning it either fully succeeds or fully fails, preventing partial writes.
*   `print(...)`: Prints a confirmation message about the atomic append.
*   `spark.read.format("delta").load(TABLE_PATH).show()`: Reads and displays the updated Delta table, showing the newly appended rows.

#### STEP 3: Schema Enforcement (Reject Malformed Ingestion)

This step illustrates Delta Lake's schema enforcement, which prevents writing data that doesn't conform to the table's defined schema, thus maintaining data quality.

```python
    sep("STEP 3 — Schema Enforcement (Reject Malformed Ingestion)")
    print("Attempting to write a row with an undeclared 'discount' column...")

    bad_schema = StructType([
        StructField("order_id",  StringType(),  True),
        StructField("user_id",   StringType(),  True),
        StructField("product",   StringType(),  True),
        StructField("price",     DoubleType(),  True),
        StructField("quantity",  IntegerType(), True),
        StructField("region",    StringType(),  True),
        StructField("discount",  DoubleType(),  True),  # Not in the registered schema
    ])
    df_bad = spark.createDataFrame(
        [("ORD999", "usr_hacker", "Free Item", 0.0, 99, "MENA", 100.0)],
        bad_schema,
    )
    try:
        df_bad.write.format("delta").mode("append").save(TABLE_PATH)
    except Exception as e:
        print("Schema violation caught by Delta Lake — write REJECTED.")
        print(f"  Error: {str(e).splitlines()[0]}")
        print("\nWhy this matters: without schema enforcement a single misconfigured")
        print("upstream pipeline can silently corrupt a production lakehouse table.")
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Explains the upcoming attempt to write data with a mismatched schema.
*   `bad_schema = StructType([...])`: Defines a new schema that includes an additional column, `discount`, which is not present in the original `ORDER_SCHEMA`.
*   `df_bad = spark.createDataFrame(...)`: Creates a DataFrame `df_bad` using this `bad_schema`.
*   `try...except Exception as e:`: This block attempts to write `df_bad` to the Delta table. Because of schema enforcement, this write operation is expected to fail.
    *   `df_bad.write.format("delta").mode("append").save(TABLE_PATH)`: Tries to append the DataFrame with the `bad_schema`.
    *   The `except` block catches the expected exception (a schema violation) and prints messages explaining why the write was rejected and the importance of schema enforcement for data integrity.

#### STEP 4: MERGE / UPSERT (Atomic Update + Insert in One Transaction)

This step showcases the powerful `MERGE` operation, which allows for atomic updates and inserts (upserts) in a single transaction, a capability often found in traditional data warehouses.

```python
    sep("STEP 4 — MERGE / UPSERT (Atomic Update + Insert in One Transaction)")
    print("""
MERGE is the most powerful Delta operation — impossible in a raw data lake:
  • If a matching record exists  → UPDATE specific columns
  • If no matching record        → INSERT the new row
  • All in one atomic transaction (either all commits or nothing commits)

Use case: a nightly CDC (Change Data Capture) feed from a production database
that may contain both price corrections and new orders in the same payload.
""")
    upsert_data = [
        ("ORD002", "usr_bob",    "Mouse",       29.99, 2, "EU"),   # exists → update price
        ("ORD010", "usr_frank",  "SSD Drive",   95.00, 1, "US"),   # new    → insert
    ]
    df_upsert = spark.createDataFrame(upsert_data, ORDER_SCHEMA)

    delta_table = DeltaTable.forPath(spark, TABLE_PATH)
    (
        delta_table.alias("target")
        .merge(
            df_upsert.alias("updates"),
            "target.order_id = updates.order_id",
        )
        .whenMatchedUpdate(set={
            "price":    "updates.price",
            "quantity": "updates.quantity",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE complete. ORD002 price updated to 29.99; ORD010 inserted.")
    spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Provides a detailed explanation of what the `MERGE` operation does and its benefits.
*   `upsert_data = [...]`: Defines data for the upsert operation. One record (`ORD002`) already exists in the table and will be updated; another (`ORD010`) is new and will be inserted.
*   `df_upsert = spark.createDataFrame(upsert_data, ORDER_SCHEMA)`: Creates a DataFrame from the upsert data.
*   `delta_table = DeltaTable.forPath(spark, TABLE_PATH)`: Creates a `DeltaTable` object, which provides an API for interacting with the Delta table.
*   `delta_table.alias("target").merge(...)`: Starts the merge operation. The existing table is aliased as `target`.
    *   `.merge(df_upsert.alias("updates"), "target.order_id = updates.order_id")`: Merges the `target` table with the `updates` DataFrame (aliased from `df_upsert`), using `order_id` as the join key.
    *   `.whenMatchedUpdate(set={...})`: Specifies actions to take when a record in `updates` *matches* a record in `target` (based on `order_id`). Here, it updates the `price` and `quantity` columns.
    *   `.whenNotMatchedInsertAll()`: Specifies actions to take when a record in `updates` *does not match* any record in `target`. Here, it inserts all columns of the new record.
    *   `.execute()`: Executes the merge operation.
*   `print(...)`: Prints a confirmation message about the merge results.
*   `spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()`: Reads and displays the updated table, sorted by `order_id` to show the changes clearly.

#### STEP 5: OPTIMIZE + Z-ORDER BY (Data Skipping for Query Acceleration)

This step demonstrates how to optimize the physical layout of the Delta table for faster query performance by consolidating small files and using Z-Ordering.

```python
    sep("STEP 5 — OPTIMIZE + Z-ORDER BY (Data Skipping for Query Acceleration)")
    print("""
The small-file problem:
  Streaming micro-batches and per-partition writes create thousands of tiny
  Parquet files. Each file read requires a separate I/O operation — at scale
  this kills query performance.

OPTIMIZE:
  Merges small files into fewer, larger Parquet files, reducing metadata
  overhead by up to 99%.

Z-ORDER BY:
  Physically co-locates rows with similar values in the same Parquet file.
  Delta reads the _delta_log/ min/max statistics per file. If your query is
  WHERE user_id = 'usr_alice', Spark skips all files that cannot contain that
  user — called data skipping.

Benchmark: Z-Ordering a 1 TB table on user_id typically reduces query time
from 4 minutes → 8 seconds on filtered queries.
""")
    delta_table.optimize().executeZOrderBy(["user_id"])
    print("OPTIMIZE + ZORDER complete.")
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Explains the

Delta Lake's `OPTIMIZE` and `Z-ORDER BY` features.

*   `delta_table.optimize().executeZOrderBy(["user_id"])`: This is the core operation for optimization and Z-ordering.
    *   `delta_table.optimize()`: Initiates the optimization process, which aims to coalesce small files into larger ones, reducing the number of files and improving read performance.
    *   `.executeZOrderBy(["user_id"])`: Applies Z-Ordering on the `user_id` column. Z-Ordering physically co-locates data records with similar `user_id` values within the same data files. This allows Delta Lake to use file-level statistics (min/max values) to quickly skip entire files that do not contain the `user_id` being queried, significantly speeding up filtered queries.
*   `print(...)`: Prints a confirmation that the optimization and Z-Ordering are complete.

#### STEP 6: Delta Transaction Log Inspection

This step demonstrates how to directly inspect the transaction log (`_delta_log/`) of a Delta table, which records every change to the table and enables ACID properties and Time Travel.

```python
    sep("STEP 6 — Inspecting the Delta Transaction Log (_delta_log/)")
    print("""
Every Delta write appends a JSON file to _delta_log/.
This is the single source of truth for ACID compliance, Time Travel, and lineage.
Each JSON file contains one or more entries:
  commitInfo  → operation type, timestamp, user, operation parameters
  add         → new Parquet file paths written in this commit
  remove      → old Parquet file paths superseded by this commit
  metaData    → schema changes (only present when schema mutates)

After every 10 commits Delta writes a checkpoint.parquet to avoid replaying
thousands of JSON files on startup — this is the Delta Log compaction process.
""")
    log_files = sorted(glob.glob(f"{TABLE_PATH}/_delta_log/*.json"))
    for log_path in log_files[:4]:
        print(f"\n--- {os.path.basename(log_path)} ---")
        with open(log_path, encoding="utf-8") as fh:
            for line in fh:
                entry = json.loads(line)
                if "commitInfo" in entry:
                    info = entry["commitInfo"]
                    print(f"  operation  : {info.get('operation', 'N/A')}")
                    print(f"  timestamp  : {info.get('timestamp', 'N/A')}")
                    params = info.get("operationParameters", {})
                    if params:
                        print(f"  parameters : {params}")
                elif "add" in entry:
                    print(f"  add file   : {entry['add'].get('path', '')}")
                elif "remove" in entry:
                    print(f"  remove file: {entry['remove'].get('path', '')}")
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Provides an explanation of the Delta Transaction Log, its structure, and its importance for ACID properties, Time Travel, and lineage.
*   `log_files = sorted(glob.glob(f"{TABLE_PATH}/_delta_log/*.json"))`: Uses the `glob` module to find all JSON files within the `_delta_log` subdirectory of the Delta table and sorts them chronologically.
*   `for log_path in log_files[:4]:`: Iterates through the first four transaction log files to demonstrate their content.
*   `print(f"\n--- {os.path.basename(log_path)} ---")`: Prints the filename of the current log file.
*   `with open(log_path, encoding="utf-8") as fh:`: Opens each JSON log file for reading.
*   `for line in fh:`: Each line in a Delta transaction log file represents a single action (e.g., adding a file, removing a file, a commit info).
*   `entry = json.loads(line)`: Parses the JSON string from each line into a Python dictionary.
*   `if "commitInfo" in entry: ...`: If the entry contains `commitInfo`, it extracts and prints details about the operation (e.g., type of operation, timestamp, parameters).
*   `elif "add" in entry: ...`: If the entry contains `add`, it indicates a new data file was added and prints its path.
*   `elif "remove" in entry: ...`: If the entry contains `remove`, it indicates an old data file was removed (e.g., due to an overwrite or update) and prints its path.

#### STEP 7: Time Travel (Query Historical Versions)

This step demonstrates Delta Lake's Time Travel capability, allowing you to query previous versions of the table by specifying a version number or a timestamp.

```python
    sep("STEP 7 — Time Travel (Query Historical Versions)")

    print("Full audit trail from the Delta Transaction Log:")
    delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

    print("Restoring state as of VERSION 0 (initial ingest only):")
    spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH).show()

    print("Restoring state as of VERSION 1 (after first ACID append):")
    spark.read.format("delta").option("versionAsOf", 1).load(TABLE_PATH).show()
```

*   `sep(...)`: Prints a separator with the step title.
*   `print("Full audit trail...")`: Informs the user that the full history will be displayed.
*   `delta_table.history().select("version", "timestamp", "operation").show(truncate=False)`: Retrieves the full history of the Delta table, showing each version, its timestamp, and the operation that created it. `truncate=False` ensures that all column content is displayed.
*   `print("Restoring state as of VERSION 0...")`: Indicates that the table is being queried as it appeared after the first commit (version 0).
*   `spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH).show()`: Reads the Delta table at a specific historical version (version 0) and displays its content. This effectively

`restores` the table to its state at version 0.
*   `print("Restoring state as of VERSION 1...")`: Indicates that the table is being queried as it appeared after the second commit (version 1).
*   `spark.read.format("delta").option("versionAsOf", 1).load(TABLE_PATH).show()`: Reads the Delta table at a specific historical version (version 1) and displays its content. This shows the state of the table after the initial batch and the first append.

#### STEP 8: VACUUM (Storage Retention Cleanup)

This step demonstrates how to use the `VACUUM` command to remove data files that are no longer referenced by the Delta table's transaction log and are older than a specified retention period.

```python
    sep("STEP 8 — VACUUM (Storage Retention Cleanup)")
    print("""
Delta retains all historical Parquet files to support Time Travel.
VACUUM permanently deletes files older than the retention window.

WARNING: After VACUUM, Time Travel beyond the retention window is impossible.

Production recommendation: retentionHours >= 168 (7 days).
Under GDPR Article 17 (right to erasure), lower retention may be required —
but confirm with your legal team before reducing below 168 hours.

Setting 0 hours here for demonstration only — NEVER do this in production.
""")
    spark.conf.set(
        "spark.databricks.delta.retentionDurationCheck.enabled", "false"
    )
    delta_table.vacuum(retentionHours=0)
    print("VACUUM complete. Orphaned files removed.")
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Provides a detailed explanation of `VACUUM`, its purpose (storage cleanup), its implications (loss of time travel history), and important considerations for production environments regarding retention periods.
*   `spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")`: This configuration setting temporarily disables the check that prevents users from setting a very low retention duration (like 0 hours). This is done here *only for demonstration purposes* to allow immediate cleanup, but it's strongly advised against in production.
*   `delta_table.vacuum(retentionHours=0)`: Executes the `VACUUM` command on the `delta_table`. `retentionHours=0` means that all files no longer part of the latest table state are immediately eligible for deletion, regardless of their age. This will permanently delete older data files, reducing storage costs but removing the ability to time travel to those specific versions.

#### STEP 9: Schema Evolution (Safe Column Addition via mergeSchema)

This step demonstrates how Delta Lake's schema evolution feature allows for safely adding new columns to a table without breaking existing data or pipelines.

```python
    sep("STEP 9 — Schema Evolution (Safe Column Addition via mergeSchema)")
    print("""
Scenario: a product team wants to add a 'discount' column to the orders feed.

Without Schema Evolution: the write would be REJECTED (Step 3 behaviour).
With mergeSchema=True:   the new column is added to the registered schema.
                         Existing rows receive NULL for the new column.

This is the production-safe migration path:
  1. Data contract teams agree on the new column definition.
  2. Producer writes first batch with mergeSchema=True.
  3. Delta propagates the schema change; all downstream readers automatically
     see the new nullable column on their next read.
""")
    evolved_schema = StructType([
        StructField("order_id",  StringType(),  True),
        StructField("user_id",   StringType(),  True),
        StructField("product",   StringType(),  True),
        StructField("price",     DoubleType(),  True),
        StructField("quantity",  IntegerType(), True),
        StructField("region",    StringType(),  True),
        StructField("discount",  DoubleType(),  True),  # new column
    ])
    df_evolved = spark.createDataFrame(
        [("ORD011", "usr_grace", "SmartWatch", 250.00, 1, "MENA", 15.0)],
        evolved_schema,
    )
    (df_evolved.write
               .format("delta")
               .mode("append")
               .option("mergeSchema", "true")
               .save(TABLE_PATH))

    print("New column 'discount' merged into existing schema without downtime.")
    print("Pre-evolution rows show NULL for 'discount' — backward compatible:")
    spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()
```

*   `sep(...)`: Prints a separator with the step title.
*   `print(...)`: Explains the concept of Schema Evolution, contrasting it with schema enforcement (from Step 3), and describes the benefits and typical workflow for safely adding new columns.
*   `evolved_schema = StructType([...])`: Defines a new schema, `evolved_schema`, which includes the existing columns plus a new `discount` column.
*   `df_evolved = spark.createDataFrame(...)`: Creates a DataFrame `df_evolved` using this new schema, including data for the new `discount` column for a new order.
*   `(df_evolved.write.format("delta").mode("append").option("mergeSchema", "true").save(TABLE_PATH))`: Appends the new DataFrame to the Delta table.
    *   `.option("mergeSchema", "true")`: This crucial option enables schema evolution. When set to `true`, if the DataFrame being written has a schema that is a superset of the table's current schema, Delta Lake will automatically evolve the table's schema to include the new columns. Existing rows will have `NULL` values for the newly added columns.
*   `print(...)`: Prints a confirmation message.
*   `spark.read.format("delta").load(TABLE_PATH).orderBy("order_id").show()`: Reads and displays the table again. This time, the `discount` column is present. Existing rows (from previous steps) will show `NULL` for `discount`, while the newly added row (`ORD011`) will have its `discount` value.

#### Final Execution Block

This block ensures the Spark session is properly terminated and confirms the completion of the lab.

```python
    spark.stop()
    print("\nMini Lakehouse lab complete. All 9 steps executed successfully.")


if __name__ == "__main__":
    main()
```

*   `spark.stop()`: Stops the SparkSession, releasing all its resources.
*   `print(...)`: Prints a final confirmation message indicating that the entire lab has completed successfully.
*   `if __name__ == "__main__": main()`: This standard Python construct ensures that the `main()` function is called only when the script is executed directly (not when it's imported as a module). This starts the entire workflow defined in the `main` function.